In [29]:
import os
os.chdir('/vast/projects/ml4chem/NikitaFedik/TPS/revision')

In [23]:
model_path = '/vast/projects/ml4chem/wei_li/test_models/quad12_GPU0_seed533257'
from hippynn.experiment.serialization import load_model_from_cwd
import torch
import hippynn

In [24]:
cwd = os.getcwd()
os.chdir(model_path)
model = load_model_from_cwd(map_location=torch.device('cpu'))
predictor = hippynn.graphs.Predictor.from_graph(model)
predictor.to(torch.float64)
os.chdir(cwd)

In [28]:
pwd

'/vast/projects/ml4chem/wei_li/test_models/quad12_GPU0_seed533257'

In [34]:
from hippynn.interfaces.ase_interface import HippynnCalculator
energy_node = model.node_from_name("T")
calc = HippynnCalculator(energy_node)
calc.to(torch.float64)

In [ ]:
from ase import units
from ase.build import molecule
from ase.md.langevin import Langevin
from ase.io.trajectory import Trajectory
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution

from ase.io import read

# Load or build the molecule (example: water molecule)
atoms = read('C7eq-TT.xyz')

# Set up a calculator (replace with your preferred calculator)
from ase.calculators.emt import EMT
atoms.calc = calc

# Set initial velocities corresponding to 300 K
MaxwellBoltzmannDistribution(atoms, temperature_K=300)

# Define the Langevin dynamics parameters
timestep = 2 * units.fs  # Time step of 1 femtosecond
temperature = 300  # Temperature in Kelvin
friction = 0.1 / units.fs # Friction coefficient in atomic units

# Initialize Langevin dynamics
dyn = Langevin(atoms, timestep, temperature_K=temperature, friction=friction)

# Attach a trajectory file to save positions during the simulation
traj = Trajectory("langevin_300K_hipnn.traj", "w", atoms)
dyn.attach(traj.write, interval=10)

#Function to print energies during the simulation
def print_energy(a=atoms):
    epot = a.get_potential_energy() / len(a)  # Potential energy per atom
    ekin = a.get_kinetic_energy() / len(a)  # Kinetic energy per atom
    temp = ekin / (1.5 * units.kB)  # Temperature from kinetic energy
    print(f"Energy per atom: Epot={epot:.3f} eV Ekin={ekin:.3f} eV (T={temp:.1f} K)")

dyn.attach(print_energy, interval=100)

# Run the dynamics for 1000 steps (adjust as needed)
print("Starting Langevin dynamics...")
print_energy()
dyn.run(10000)

Starting Langevin dynamics...
Energy per atom: Epot=-4.377 eV Ekin=0.036 eV (T=280.1 K)
Energy per atom: Epot=-4.377 eV Ekin=0.036 eV (T=280.1 K)
Energy per atom: Epot=-4.347 eV Ekin=0.035 eV (T=271.3 K)
Energy per atom: Epot=-4.338 eV Ekin=0.035 eV (T=271.8 K)
Energy per atom: Epot=-4.347 eV Ekin=0.037 eV (T=285.8 K)
Energy per atom: Epot=-4.340 eV Ekin=0.035 eV (T=269.8 K)
Energy per atom: Epot=-4.337 eV Ekin=0.040 eV (T=310.8 K)
Energy per atom: Epot=-4.342 eV Ekin=0.035 eV (T=270.9 K)
Energy per atom: Epot=-4.329 eV Ekin=0.027 eV (T=205.8 K)
Energy per atom: Epot=-4.351 eV Ekin=0.053 eV (T=413.5 K)
Energy per atom: Epot=-4.335 eV Ekin=0.033 eV (T=256.9 K)
Energy per atom: Epot=-4.348 eV Ekin=0.036 eV (T=277.7 K)


True

In [ ]:
# TODO: figure out units: 

In [16]:
import nglview as nv

# Create a simple visualization
view = nv.show_ase(atoms)  # Replace with your PDB file path
view

NGLWidget()